In [1]:
import json
import requests
import re
import sys
import pickle
import time
RIS_PREFIXLIST_URL = 'https://stat.ripe.net/data/announced-prefixes/data.json'
def get_prefix(asn):
    try:
        #print(asn)
        rslt = requests.get(RIS_PREFIXLIST_URL, {
            'resource': asn
        }).json()
        # 尝试获取前缀
        prefix = rslt['data']['prefixes'][0]['prefix']
        return prefix
    except IndexError:
        print(asn,'IndexError')
        # 当列表索引越界时，返回当前的rslt
        return None
    except Exception as e:
        # 其他异常也可以返回rslt或错误信息（可选）
        print(asn,f"发生其他错误: {e}")
        return None  # 或根据需要返回其他标识，如None

def load_topology_data(filename: str) -> dict:
    """从文件加载拓扑数据（兼容 key: value 格式）"""
    with open(filename, 'r') as f:
        content = f.read().strip()
    
    # 处理格式：添加外层大括号，替换冒号为冒号+引号，处理列表
    # 1. 替换 key: 为 'key':
    content = re.sub(r'^(\w+):', r'"\1":', content, flags=re.MULTILINE)
    # 2. 每行末尾添加逗号（最后一行除外）
    lines = content.split('\n')
    lines = [line + ',' for line in lines[:-1]] + [lines[-1]] if lines else []
    content = '\n'.join(lines)
    # 3. 包裹成字典
    content = '{' + content + '}'
    
    # 安全解析为字典
    try:
        return eval(content)  # 此处使用eval是因为处理后的格式已符合Python字典规范
    except Exception as e:
        raise ValueError(f"解析拓扑数据失败: {e}")

t=time.time()
try:
    TOPOLOGY_DATA = load_topology_data('real_topology.txt')
except FileNotFoundError:
    print("错误: 未找到real_topology.txt文件")
    sys.exit(1)
    
# prefix_dict = {}
# for asn in TOPOLOGY_DATA['transit_asns']:
#     asn_int = int(asn)  # 转换为整数ASN
#     result = get_prefix(asn_int)
    
#     # if isinstance(result, dict):
#     #     raise ValueError(f"获取ASN {asn_int} 的前缀失败，返回了JSON数据: {result}")
    
#     # 若不是字典，则认为是有效前缀（字符串类型），存入字典
#     prefix_dict[asn_int] = result

# print(f"获取前缀耗时: {time.time() - t} 秒")
# with open("my_dict.pkl", "wb") as f:
#     pickle.dump(prefix_dict, f)
# print("字典已成功存储为 my_dict.pkl")

In [2]:
# import json
# import requests
# import re
# import sys
# import pickle
# import time
# RIS_PREFIXLIST_URL = 'https://stat.ripe.net/data/announced-prefixes/data.json'
# rslt = requests.get(RIS_PREFIXLIST_URL, {
#         'resource': 30740
#     }).json()
# prefix=rslt['data']['prefixes'][0]['prefix']
# print(rslt['data'])

In [3]:
# with open("my_dict.pkl", "rb") as f:
#     prefix_dict = pickle.load(f)
# print(f"字典已成功加载为 prefix_dict，共包含 {len(prefix_dict)} 条记录")

In [4]:
# prefix_dict['30740']=prefix

In [5]:
# for key,value in prefix_dict.items():
#     print(f"{key}: {value}")

### 有些是ipv6地址，有些是没有prefix，这个之后再处理

In [6]:
def assign_asn_and_ip(ixps, transit_asns):
    """
    为ixps和transit_asns列表中的所有元素分配ASN和IPv4地址
    
    参数:
        ixps (list): 包含IXP名称的字符串列表
        transit_asns (list): 包含Transit AS名称的字符串列表
    
    返回:
        dict: 键为原始字符串，值为包含"asn"和"ipv4"的字典
    """
    # 合并两个列表，保持原有顺序（先ixps后transit_asns）
    all_items = ixps + transit_asns
    assignment = {}
    
    for index, item in enumerate(all_items, start=1):
        asn = index  # ASN从1开始按顺序分配
        # IPv4前16位为ASN，后16位从1开始递增（每个item唯一）
        # 转换为点分十进制格式（前16位拆分为前两个字节，后16位拆分为后两个字节）
        ip_first_part = asn //256  # 前8位
        ip_second_part = asn %256  # 后8位（前16位的低8位）
        ip_third_part = 0  # 后16位的高8位（固定为0，可根据需要调整）
        ip_fourth_part = 0  # 后16位的低8位（从1开始）
        
        ipv4 = f"{ip_first_part}.{ip_second_part}.{ip_third_part}.{ip_fourth_part}/16"
        assignment[item] = {
            "asn": asn,
            "ipv4": ipv4
        }
    
    return assignment
assignment=assign_asn_and_ip(TOPOLOGY_DATA['ixps'], TOPOLOGY_DATA['transit_asns'])
print(assignment)

{'Newark-DE-US': {'asn': 1, 'ipv4': '0.1.0.0/16'}, 'Boston-MA-US': {'asn': 2, 'ipv4': '0.2.0.0/16'}, 'San Jose-CA-US': {'asn': 3, 'ipv4': '0.3.0.0/16'}, 'Phoenix-AZ-US': {'asn': 4, 'ipv4': '0.4.0.0/16'}, 'Houston-TX-US': {'asn': 5, 'ipv4': '0.5.0.0/16'}, 'New York City-NY-US': {'asn': 6, 'ipv4': '0.6.0.0/16'}, 'Dallas-TX-US': {'asn': 7, 'ipv4': '0.7.0.0/16'}, 'Huntsville-AL-US': {'asn': 8, 'ipv4': '0.8.0.0/16'}, 'Palo Alto-CA-US': {'asn': 9, 'ipv4': '0.9.0.0/16'}, 'Seattle-WA-US': {'asn': 10, 'ipv4': '0.10.0.0/16'}, 'Stamford-CT-US': {'asn': 11, 'ipv4': '0.11.0.0/16'}, 'London-ENG-UK': {'asn': 12, 'ipv4': '0.12.0.0/16'}, 'Frankfurt Am Main-05-DE': {'asn': 13, 'ipv4': '0.13.0.0/16'}, 'Berlin-16-DE': {'asn': 14, 'ipv4': '0.14.0.0/16'}, 'Mascot-02-AU': {'asn': 15, 'ipv4': '0.15.0.0/16'}, 'Chicago-IL-US': {'asn': 16, 'ipv4': '0.16.0.0/16'}, 'Changi-04-SG': {'asn': 17, 'ipv4': '0.17.0.0/16'}, 'Los Angeles-CA-US': {'asn': 18, 'ipv4': '0.18.0.0/16'}, 'Atlanta-GA-US': {'asn': 19, 'ipv4': '0.19

In [ ]:
with open("assignment.pkl", "wb") as f:
    pickle.dump(assignment, f)
# with open("assignment.pkl", "rb") as f:
#     assignment = pickle.load(f)